1. Import the necessary packages

In [82]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk

2. loading the data set 
These are the order of the df.iloc numerical column index 

- ID  age  ho_incm  edu  BD1  BD1_11  BM7  BM14  O_TC  BS1_1

- starting from '300' is the additional smoking features used just for testing the NaN count to replace 'BS3_1' and 'BS3_2' 


In [83]:
df = pd.read_csv('KNHANES9.csv') 

df_subset = (df.iloc[:, [1, 9, 12, 15, 232, 234, 300, 520, 535, 496]])
age_threshold = 50 
df_subset = df_subset[df_subset['age'] >= age_threshold]
print(df_subset.head(1))

           ID  age ho_incm edu BD1 BD1_11 BS1_1 BM7 BM14 O_TC
5  WA01264601   80       1   1   1      8     3   2    1   23


/var/folders/xt/mzqh8hz12q76z91q0t6kvzqm0000gn/T/ipykernel_84509/940809374.py:1: DtypeWarning: Columns (11,12,13,14,24,25,27,28,29,30,31,32,33,34,35,36,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,63,64,65,66,67,69,70,71,72,73,74,75,76,78,79,80,81,82,83,84,85,86,87,88,89,90,91,93,94,95,96,97,98,99,100,101,102,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,134,135,136,137,138,139,140,141,142,144,145,146,147,148,149,150,151,152,153,155,156,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,249,250,251,252,253,254,255,256,257,258,259,260,261,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,283,284,285,286,287,288,289,291,292,294,295,296,297,298,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,

3. 
Counting the values of NaN etc.
a. it shows which values it will drop, considers which are NaN 
^ the value that its going to drop is ['-1.80E+308'] <-- classified as NaN  in the data set 
b. 'errors = 'coerce'' means that its going to take those ['-1.80E+308'] and 
    swap them to 'NaN' 
c. removes all rows where O_TC is NaN, keeps rows with valid tooth count
subset=['O_TC'] means that it only checks that column --> drops it / deletes
d.  creates a new binary label column for O_TC; 
    'astype(int) <-- computer views 1 = True; 0 = False 
    'O_TC_' <-- creates a new column 
e. 'label_count' = counts how many people are in each group 0 and 1 


In [84]:
print(df_subset[pd.to_numeric(df_subset['O_TC'], errors='coerce').isna()]['O_TC'].unique())

df_subset['O_TC'] = pd.to_numeric(df_subset['O_TC'], errors='coerce')
before_drop = len(df_subset)
df_subset = df_subset.dropna(subset=['O_TC'])
print(f"Rows droppped (NaN O_TC): {before_drop - len(df_subset)}")

df_subset['O_TC_'] = (df_subset['O_TC'] < 20).astype(int)

label_counts    = df_subset['O_TC_'].value_counts()
label_0         = label_counts[0]
label_1         = label_counts[1]
imbalance_ratio = label_0 / label_1

print(f"Age filter: >= {age_threshold}")
print(f"Label 0 (≥20 teeth): {label_0}")
print(f"Label 1 (<20 teeth): {label_1}")
print(df_subset['O_TC'].value_counts(dropna=False).head(20))
print(f"Imbalance ratio: {imbalance_ratio:.2f}")

print(f"NaN count: {df_subset['O_TC'].isna().sum()}")

['-1.80E+308']
Rows droppped (NaN O_TC): 410
Age filter: >= 50
Label 0 (≥20 teeth): 2441
Label 1 (<20 teeth): 894
O_TC
28.0    556
27.0    350
26.0    283
25.0    188
0.0     179
24.0    179
23.0    166
32.0    132
29.0    128
21.0    114
22.0    112
30.0     95
20.0     89
19.0     67
16.0     59
17.0     53
18.0     49
31.0     49
15.0     41
12.0     39
Name: count, dtype: int64
Imbalance ratio: 2.73
NaN count: 0


4. Missing Value Audit 

In [85]:
invalid_codes = [8, 9, 88, 99, 888, 999, 8888, 9999]
feature_cols  = ['age', 'ho_incm', 'edu', 'BD1', 'BD1_11', 'BS1_1', 'BM7', 'BM14']


df_subset[feature_cols] = df_subset[feature_cols].replace(invalid_codes, np.nan)
df_subset[feature_cols] = df_subset[feature_cols].apply(pd.to_numeric, errors='coerce')
df_subset[feature_cols] = df_subset[feature_cols].where(df_subset[feature_cols].abs() < 1e10, np.nan)

print("NaN counts per column after cleaning:")
total = len(df_subset)
for col in feature_cols:
    count = df_subset[col].isna().sum()
    pct = (count/total) * 100 
    print(f"{col}: {count} | ({pct:.1f}%)")




NaN counts per column after cleaning:
age: 0 | (0.0%)
ho_incm: 22 | (0.7%)
edu: 137 | (4.1%)
BD1: 49 | (1.5%)
BD1_11: 205 | (6.1%)
BS1_1: 49 | (1.5%)
BM7: 49 | (1.5%)
BM14: 49 | (1.5%)


5. TRAIN TEST SPLIT

In [86]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

X = df_subset[['age', 'ho_incm', 'edu', 'BD1', 'BD1_11', 'BS1_1', 'BM7', 'BM14']]
y = df_subset['O_TC_']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

X_train: (2668, 8), X_test: (667, 8)


6. Simple Imputer

In [87]:
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

7. Standardization 

In [88]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# For Logistic Regression and SVM — normalize all features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# For RF, XGBoost, LightGBM — NO standardization needed
X_train_tree = X_train.copy()
X_test_tree  = X_test.copy()

8. Class Imbalance -- using SMOTE 

In [89]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy = 1/1.5, k_neighbors=5, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", pd.Series(y_train).value_counts().to_dict())
print("After SMOTE: ", pd.Series(y_train_resampled).value_counts().to_dict())

Before SMOTE: {0: 1953, 1: 715}
After SMOTE:  {0: 1953, 1: 1302}


9. Hyperparameter Tuning

In [92]:
import sys
!{sys.executable} -m pip install xgboost

  Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-macosx_12_0_arm64.whl (2.3 MB)

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [93]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import pandas as pd

param_grids = {
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators':     [100, 300, 500],
            'max_depth':        [5, 10, 15, None],
            'min_samples_leaf': [1, 5, 10]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss'),
        'params': {
            'n_estimators':  [100, 300, 500],
            'max_depth':     [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.3],
            'subsample':     [0.7, 0.8, 1.0]
        }
    },
    'LightGBM': {
        'model': LGBMClassifier(random_state=42),
        'params': {
            'n_estimators':     [100, 300, 500],
            'num_leaves':       [15, 31, 63],
            'learning_rate':    [0.01, 0.1, 0.3],
            'min_child_samples':[5, 10, 20]
        }
    },
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000, solver='saga'),
        'params': {
            'C':       [0.01, 0.1, 1, 10],
            'penalty': ['l1', 'l2']
        }
    },
    'SVM': {
        'model': SVC(probability=True, random_state=42),
        'params': {
            'C':      [0.1, 1, 10],
            'kernel': ['rbf', 'linear'],
            'gamma':  ['scale', 'auto']
        }
    }
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, config in param_grids.items():
    search = RandomizedSearchCV(
        estimator          = config['model'],
        param_distributions= config['params'],
        n_iter             = 50,
        cv                 = cv,
        scoring            = 'roc_auc',
        random_state       = 42,
        n_jobs             = -1
    )
    search.fit(X_train_resampled, y_train_resampled)
    results[name] = {
        'best_params':    search.best_params_,
        'best_cv_score':  search.best_score_
    }
    print(f"{name}: Best AUC = {search.best_score_:.4f} | Params: {search.best_params_}")

# Print comparison table ranked by AUC
results_df = pd.DataFrame(
    {name: {'AUC': v['best_cv_score']} for name, v in results.items()}
).T.sort_values('AUC', ascending=False)
print(results_df)

/Users/yonghan/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 36 is smaller than n_iter=50. Running 36 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Random Forest: Best AUC = 0.8697 | Params: {'n_estimators': 500, 'min_samples_leaf': 1, 'max_depth': 10}
XGBoost: Best AUC = 0.8741 | Params: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
[LightGBM] [Info] Number of positive: 1042, number of negative: 1562
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000462 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 484
[LightGBM] [Info] Number of data points in the train set: 2604, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.400154 -> initscore=-0.404825
[LightGBM] [Info] Start training from score -0.404825
[LightGBM] [Info] Number of positive: 1041, number of negative: 1563
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000400 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 462
[LightGBM] [Info] Number of

/Users/yonghan/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 8 is smaller than n_iter=50. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
/Users/yonghan/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/yonghan/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of pe

Logistic Regression: Best AUC = 0.7906 | Params: {'penalty': 'l2', 'C': 0.1}
SVM: Best AUC = 0.8190 | Params: {'kernel': 'rbf', 'gamma': 'auto', 'C': 10}
                          AUC
XGBoost              0.874129
LightGBM             0.872094
Random Forest        0.869729
SVM                  0.818993
Logistic Regression  0.790563
